# Somnotate Testing
Run Somnotate on annotated .mat files in data/hypnose_eeg/somnotate_testing,
then visualize scores against manual annotations and compute agreements.

In [3]:
from pathlib import Path

from utils.testing import (
    prepare_testing_csvs,
    load_manual_vectors_from_csvs,
    load_somnotate_predictions,
    save_somnotate_predictions,
    score_somnotate,
    align_vectors,
    plot_testing_comparison,
    agreement_matrix,
)

# Optional: enable an interactive backend (uncomment if needed).
%matplotlib qt

model_name = "test_model_1"
current_test = "sub_53_comparison"
repo_root = Path.cwd().parent
testing_mat_dir = repo_root / "data" / "hypnose_eeg" / "somnotate_testing" / current_test
model_path = (
    repo_root
    / "data"
    / "hypnose_eeg"
    / "derivatives"
    / "somnotate_training"
    / model_name
    / "model.pickle"
)

print("testing_mat_dir:", testing_mat_dir)
print("model_path:", model_path)

testing_mat_dir: /Users/joschua/repos/harris_lab/eeg_preprocessing/data/hypnose_eeg/somnotate_testing/sub_53_comparison
model_path: /Users/joschua/repos/harris_lab/eeg_preprocessing/data/hypnose_eeg/derivatives/somnotate_training/test_model_1/model.pickle


In [5]:
csv_dir = prepare_testing_csvs(
    testing_mat_dir=testing_mat_dir,
    repo_root=repo_root,
    model_name=model_name,
    test_name=current_test,
)

csv_files = sorted([p for p in csv_dir.glob("*.csv") if not p.name.startswith("._")])
print("CSV files:", len(csv_files))
for path in csv_files:
    print(" -", path.name)

predictions_dir = (
    repo_root
    / "data"
    / "hypnose_eeg"
    / "derivatives"
    / "somnotate_testing"
    / model_name
    / current_test
    / "somnotate_predictions"
)

reference_csv = csv_files[0] if csv_files else None
if reference_csv is None:
    raise ValueError("No CSV files found in testing output")

somnotate_pred_path, _ = save_somnotate_predictions(
    reference_csv,
    model_path,
    predictions_dir,
)
print("Somnotate predictions:", somnotate_pred_path)

Found .mat files: ['/Users/joschua/repos/harris_lab/eeg_preprocessing/data/hypnose_eeg/somnotate_testing/sub_53_comparison/sub-006_ses-01_recording-02_Nam.mat', '/Users/joschua/repos/harris_lab/eeg_preprocessing/data/hypnose_eeg/somnotate_testing/sub_53_comparison/sub-006_ses-01_recording-02_Joschua.mat', '/Users/joschua/repos/harris_lab/eeg_preprocessing/data/hypnose_eeg/somnotate_testing/sub_53_comparison/sub-006_ses-01_recording-02_Volkan.mat']
Processing file: /Users/joschua/repos/harris_lab/eeg_preprocessing/data/hypnose_eeg/somnotate_testing/sub_53_comparison/sub-006_ses-01_recording-02_Nam.mat
EEG1 data extracted successfully.
EEG2 data extracted successfully.
EMG data extracted successfully.
Sleep stage data extracted successfully.
Length of upsampled sleep stages (17981440) does not match length of EEG data (17981441) by 1 samples
Upsampled sleep stages padded with zeros to match length of EEG data
Length of upsampled sleep stages matches length of EEG data after truncation
Sa

In [5]:
# Pick one or more CSVs to compare (each file can contain one or more sleepStage columns).
csv_paths = "all"  # use "all" or a list like [csv_files[0], csv_files[1]]

if csv_paths == "all":
    csv_paths = csv_files

raw_signals, manual_vectors = load_manual_vectors_from_csvs(csv_paths)
somnotate_vec = load_somnotate_predictions(somnotate_pred_path)
somnotate_vec, manual_vectors = align_vectors(somnotate_vec, manual_vectors)

fig, viewer = plot_testing_comparison(
    raw_signals,
    sampling_rate_hz=512,
    somnotate_vec=somnotate_vec,
    manual_vectors=manual_vectors,
 )

NameError: name 'csv_files' is not defined

In [ ]:
from pathlib import Path
import numpy as np

# Reload minimal inputs if this cell is run standalone.
repo_root = Path.cwd().parent
testing_root = repo_root / "data" / "hypnose_eeg" / "derivatives" / "somnotate_testing"
current_test = locals().get("current_test", "sub_53_comparison")
model_name = locals().get("model_name", "test_model_1")
test_root = testing_root / model_name / current_test

somnotate_pred_path = locals().get(
    "somnotate_pred_path",
    test_root / "somnotate_predictions" / "somnotate_predictions.csv",
)

if "somnotate_vec" not in locals() and somnotate_pred_path.exists():
    somnotate_vec = load_somnotate_predictions(somnotate_pred_path)

if "manual_vectors" not in locals():
    csv_dir = test_root / "intermediate" / "csv"
    csv_files = sorted([p for p in csv_dir.glob("*.csv") if not p.name.startswith("._")])
    if not csv_files:
        raise ValueError("No CSV files found to compute agreement")
    raw_signals, manual_vectors = load_manual_vectors_from_csvs(csv_files)

if "somnotate_vec" not in locals():
    raise ValueError("Somnotate predictions not found. Run Cell 3 to generate them.")

somnotate_vec, manual_vectors = align_vectors(somnotate_vec, manual_vectors)

agreement_df = agreement_matrix(somnotate_vec, manual_vectors)
agreement_df

agreement_plot_path = test_root / "agreement_matrix.png"

from utils.testing import plot_agreement_matrix
plot_agreement_matrix(agreement_df, output_path=agreement_plot_path)
print("Saved agreement matrix:", agreement_plot_path)

NameError: name 'somnotate_vec' is not defined